<a href="https://colab.research.google.com/github/jtlaplante11-star/AAI2025/blob/2026fall/ML/house_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Part 1: Predict house prices based on square footage and location.
Author: Jacob Laplante

DATA SOURCE (real data):
  Ames Housing Dataset (De Cock, D., 2011, "Ames, Iowa: Alternative to the
  Boston Housing Data as an End of Semester Regression Project", Journal of
  Statistics Education, 19(3)).
  Official public download (no login required):
    https://jse.amstat.org/v19n3/decock/AmesHousing.txt
  Data dictionary:
    https://jse.amstat.org/v19n3/decock/DataDocumentation.txt
  2,930 home sales in Ames, IA (2006-2010), 82 columns. We use:
    - 'Gr Liv Area'  -> square_footage
    - 'Neighborhood' -> location   (restricted to the 6 most common
                                     neighborhoods so the OneHotEncoder
                                     output stays readable, same idea as
                                     the toy 'Downtown/Suburb/Rural'
                                     categories in the original starter code)
    - 'SalePrice'    -> price (target)

  NOTE ON THE SANDBOX THIS WAS DEVELOPED IN: the environment used to write
  and test this script has outbound internet access restricted to a small
  allow-list (package registries only), so the live download above could
  not be exercised there. The loader below still tries the real URL first
  (this is exactly what will run in Google Colab, GitHub Actions, or your
  own laptop, all of which have normal internet access). If the download
  is unavailable for any reason, it automatically falls back to a
  synthetic dataset generated to match the *real* Ames summary statistics
  (mean SalePrice ~ $180,796, mean Gr Liv Area ~ 1,515 sq ft, ~$100-120
  marginal price per square foot) so the script always runs end to end.
"""

import io
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error

AMES_URL = "https://jse.amstat.org/v19n3/decock/AmesHousing.txt"
TOP_N_NEIGHBORHOODS = 6
RANDOM_STATE = 42


def load_real_ames_data(top_n=TOP_N_NEIGHBORHOODS):
    """Download the real Ames Housing dataset and trim it down to the
    columns/categories this assignment needs."""
    raw = pd.read_csv(AMES_URL, sep="\t")
    df = raw[["Gr Liv Area", "Neighborhood", "SalePrice"]].rename(
        columns={
            "Gr Liv Area": "square_footage",
            "Neighborhood": "location",
            "SalePrice": "price",
        }
    ).dropna()

    top_locations = df["location"].value_counts().nlargest(top_n).index
    df = df[df["location"].isin(top_locations)].reset_index(drop=True)
    return df


def generate_fallback_data(n=300, seed=RANDOM_STATE):
    """Synthetic stand-in that mimics the real Ames Housing distribution,
    used only if the live download above is not reachable. Parameters are
    based on the dataset's published summary statistics (see module
    docstring) rather than being arbitrary."""
    rng = np.random.default_rng(seed)

    neighborhoods = {
        # name: (avg base price, price premium per sqft)
        "NAmes":   (145000, 95),
        "CollgCr": (200000, 110),
        "OldTown": (125000, 85),
        "Edwards": (130000, 90),
        "Somerst": (225000, 120),
        "Gilbert": (190000, 105),
    }
    names = list(neighborhoods.keys())[:TOP_N_NEIGHBORHOODS]

    locations = rng.choice(names, size=n)
    square_footage = rng.normal(1515, 500, size=n).clip(600, 4200).round().astype(int)

    base_price = np.array([neighborhoods[loc][0] for loc in locations])
    per_sqft = np.array([neighborhoods[loc][1] for loc in locations])
    noise = rng.normal(0, 18000, size=n)

    price = (base_price + per_sqft * square_footage + noise).clip(60000, 600000).round(-2)

    return pd.DataFrame({
        "square_footage": square_footage,
        "location": locations,
        "price": price,
    })


def load_data():
    try:
        df = load_real_ames_data()
        print(f"Loaded {len(df)} REAL home sales from the Ames Housing dataset "
              f"({AMES_URL}).")
        return df
    except Exception as exc:
        print(f"Could not download the real Ames Housing dataset ({exc}). "
              f"Falling back to a documented synthetic dataset that matches "
              f"its real summary statistics.")
        df = generate_fallback_data()
        print(f"Generated {len(df)} synthetic rows as a fallback.")
        return df


df = load_data()

# Features and target
X = df[["square_footage", "location"]]
y = df["price"]

# Preprocessing: One-hot encode the location column, pass square_footage through
preprocessor = ColumnTransformer(
    transformers=[
        ("location", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), ["location"])
    ],
    remainder="passthrough",
)

# Pipeline with preprocessing and model
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression()),
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# Train model
model.fit(X_train, y_train)

# --- Evaluation (improvement over the starter code, which never checked
# how good the model actually is) ---
y_pred_test = model.predict(X_test)
print(f"\nTest R^2:  {r2_score(y_test, y_pred_test):.3f}")
print(f"Test MAE:  ${mean_absolute_error(y_test, y_pred_test):,.0f}")

# Make a prediction for a new house: 2000 sq ft in the most common location.
# NOTE: the assignment's original example asks for a prediction in
# 'Downtown'. Since we upgraded to the real Ames Housing dataset, there is
# no neighborhood literally named 'Downtown' (real neighborhoods are named
# things like 'NAmes', 'CollgCr', etc.) -- so this predicts for the
# dataset's most common neighborhood as the representative example. You can
# predict for any of the neighborhoods used by changing `location` below.
example_location = df["location"].value_counts().idxmax()
new_house = pd.DataFrame({"square_footage": [2000], "location": [example_location]})
predicted_price = model.predict(new_house)

print(f"\nPredicted price for a 2000 sq ft house in {example_location} "
      f"(this dataset's real-data equivalent of the starter code's 'Downtown' example): "
      f"${predicted_price[0]:,.2f}")

# Display model coefficients
feature_names = (
    model.named_steps["preprocessor"]
    .named_transformers_["location"]
    .get_feature_names_out(["location"])
).tolist() + ["square_footage"]

coefficients = model.named_steps["regressor"].coef_
print("\nModel Coefficients:")
for feature, coef in zip(feature_names, coefficients):
    print(f"  {feature}: {coef:,.2f}")
print(f"  intercept: {model.named_steps['regressor'].intercept_:,.2f}")

# --- Plain-language explanation (required by the assignment) ---
sqft_coef = coefficients[-1]
location_coefs = {name.split("_", 1)[-1]: coef for name, coef in zip(feature_names[:-1], coefficients[:-1])}
priciest_loc, priciest_val = max(location_coefs.items(), key=lambda kv: kv[1])
cheapest_loc, cheapest_val = min(location_coefs.items(), key=lambda kv: kv[1])

print("\nExplanation:")
print(f"  - Square footage coefficient (${sqft_coef:,.2f}/sqft): holding the neighborhood fixed, each "
      f"additional square foot of living area is associated with about ${sqft_coef:,.2f} more in predicted "
      f"sale price. This is the model's estimate of the '$/sqft' effect the assignment describes.")
print(f"  - Location effect: each neighborhood's coefficient shows how much more (positive) or less "
      f"(negative) homes there sell for compared to other neighborhoods in the dataset, at the same square "
      f"footage. Here, '{priciest_loc}' carries the largest premium (${priciest_val:,.0f}), while "
      f"'{cheapest_loc}' carries the largest discount (${cheapest_val:,.0f}) -- so location shifts the price "
      f"up or down by a fixed dollar amount on top of the per-square-foot effect.")

Loaded 1491 REAL home sales from the Ames Housing dataset (https://jse.amstat.org/v19n3/decock/AmesHousing.txt).

Test R^2:  0.747
Test MAE:  $27,066

Predicted price for a 2000 sq ft house in NAmes (this dataset's real-data equivalent of the starter code's 'Downtown' example): $199,217.13

Model Coefficients:
  location_CollgCr: 10,692.58
  location_Edwards: -46,526.72
  location_NAmes: -30,451.25
  location_NridgHt: 98,830.60
  location_OldTown: -62,204.00
  location_Somerst: 29,658.79
  square_footage: 77.26
  intercept: 75,155.37

Explanation:
  - Square footage coefficient ($77.26/sqft): holding the neighborhood fixed, each additional square foot of living area is associated with about $77.26 more in predicted sale price. This is the model's estimate of the '$/sqft' effect the assignment describes.
  - Location effect: each neighborhood's coefficient shows how much more (positive) or less (negative) homes there sell for compared to other neighborhoods in the dataset, at the same s